In [1]:
from typing import Annotated
from langchain_groq import ChatGroq
from langchain_core.messages import AnyMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage

In [2]:
load_dotenv()

True

In [3]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [4]:

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [5]:
def chat_node(state: ChatState):

    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question.",
        "question": state["messages"][-1].content,
        "instruction": "Approve this question? yes/no"
    })
    
    if decision["approved"] == 'no':
        return {"messages": [AIMessage(content="Not approved.")]}

    else:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

In [6]:
# 3. Build the graph: START -> chat -> END
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Checkpointer is required for interrupts
checkpointer = MemorySaver()

# Compile the app
app = builder.compile(checkpointer=checkpointer)

In [7]:
# Create a new thread id for this conversation
config = {"configurable": {"thread_id": '1234'}}

# ---- STEP 1: user asks a question ----
initial_input = {
    "messages": [
        ("user", "Explain gradient descent in very simple terms.")
    ]
}

# Invoke the graph for the first time
result = app.invoke(initial_input, config=config)

In [8]:
result

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='7597e9a2-b957-4e45-af0b-a58bbee4de45')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer a user question.', 'question': 'Explain gradient descent in very simple terms.', 'instruction': 'Approve this question? yes/no'}, id='96601458124f6a6728f38c0c8ef22c94')]}

In [9]:

message = result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'Explain gradient descent in very simple terms.',
 'instruction': 'Approve this question? yes/no'}

In [10]:

user_input = input(f"\nBackend message - {message} \n Approve this question? (y/n): ")

In [11]:
# Resume the graph with the approval decision
final_result = app.invoke(
    Command(resume={"approved": user_input}),
    config=config,
)

In [12]:
print(final_result["messages"][-1].content)

**Gradient descent = “roll a ball downhill until it stops.”**

---

### 1. What are we trying to do?

Imagine you have a landscape that represents a *function* you want to minimize.  
- The **height** of the landscape at each point is the value of the function.  
- The **lowest point** (the bottom of the valley) is the answer we’re looking for.

In math we call the function **f(x)**.  
- **x** is the “position” in the landscape (it could be one number, a vector of many numbers, etc.).  
- We want to find the **x** that makes **f(x)** as small as possible.

---

### 2. How do we know which way is downhill?

At any point on the landscape you can look at the slope (the steepness).  
- In calculus the slope is called the **gradient** (or derivative).  
- The gradient points **upward**: it tells you the direction of the steepest climb.  
- To go **down** you go in the opposite direction, i.e. **negative gradient**.

---

### 3. The simple rule

1. **Pick a starting point** `x₀` (anywhere on